# Processed NPZ quick analysis

This notebook gives a compact first look at `Demo_binary/data/processed/*.npz` files: keys, shapes, dtypes, label counts, trial axis inference, and simple signal plots.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_DIR = Path('data/processed')
if not DATA_DIR.exists():
    DATA_DIR = Path('Demo_binary/data/processed')
files = sorted(DATA_DIR.glob('*.npz'))
files

In [ ]:
rows = []

for path in files:
    with np.load(path, allow_pickle=False) as data:
        keys = list(data.files)
        y = data['y'] if 'y' in keys else None
        for key in keys:
            arr = data[key]
            row = {
                'file': path.name,
                'key': key,
                'shape': arr.shape,
                'dtype': arr.dtype,
                'size': arr.size,
            }
            if np.issubdtype(arr.dtype, np.number) and arr.size:
                row.update({
                    'min': float(np.nanmin(arr)),
                    'mean': float(np.nanmean(arr)),
                    'std': float(np.nanstd(arr)),
                    'max': float(np.nanmax(arr)),
                })
            if key == 'X' and y is not None:
                matching_axes = [i for i, n in enumerate(arr.shape) if n == len(y)]
                row['axes_matching_y_len'] = matching_axes
            rows.append(row)

summary = pd.DataFrame(rows)
summary

In [ ]:
label_rows = []

for path in files:
    with np.load(path, allow_pickle=False) as data:
        y = data['y']
        values, counts = np.unique(y, return_counts=True)
        for value, count in zip(values, counts):
            label_rows.append({'file': path.name, 'label': int(value), 'count': int(count)})

label_counts = pd.DataFrame(label_rows)
label_counts.pivot(index='file', columns='label', values='count').fillna(0).astype(int)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
label_counts.pivot(index='file', columns='label', values='count').plot(kind='bar', ax=ax)
ax.set_xlabel('file')
ax.set_ylabel('trial count')
ax.set_title('Label distribution per file')
ax.legend(title='label')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()

## Inspect one file

The current files have `X` shaped like `(time, channels, trials)`, because `len(y)` matches the last axis.

In [ ]:
file_idx = 0
trial_idx = 0
channel_idx = 0

path = files[file_idx]
with np.load(path, allow_pickle=False) as data:
    X = data['X']
    y = data['y']

print(path.name)
print('X:', X.shape, X.dtype)
print('y:', y.shape, y.dtype)
print('trial label:', y[trial_idx])

In [ ]:
signal = X[:, channel_idx, trial_idx]

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(signal, lw=0.8)
ax.set_title(f'{path.name} | trial={trial_idx}, channel={channel_idx}, label={y[trial_idx]}')
ax.set_xlabel('sample')
ax.set_ylabel('amplitude')
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for label in sorted(np.unique(y)):
    class_signal = X[:, channel_idx, y == label].mean(axis=1)
    ax.plot(class_signal, lw=1.0, label=f'label {label}')

ax.set_title(f'Class mean waveform | {path.name}, channel={channel_idx}')
ax.set_xlabel('sample')
ax.set_ylabel('mean amplitude')
ax.legend()
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, len(files), figsize=(4 * len(files), 3), sharey=True)

if len(files) == 1:
    axes = [axes]

for ax, path in zip(axes, files):
    with np.load(path, allow_pickle=False) as data:
        X = data['X']
    channel_std = X.std(axis=(0, 2))
    ax.plot(channel_std, lw=0.8)
    ax.set_title(path.stem)
    ax.set_xlabel('channel')

axes[0].set_ylabel('std over time/trials')
plt.tight_layout()

## Inspect the source XDF markers

This checks `/home/jihoon/workspace/Demo_binary/260602_sub1_hjlee.xdf` directly. The preprocessing script currently treats marker values `"1"`-`"5"` as trial onsets, while the file also contains `"101"`-`"105"` markers.

In [ ]:
from collections import Counter

import pyxdf

XDF_PATH = Path('260602_sub1_hjlee.xdf')
if not XDF_PATH.exists():
    XDF_PATH = Path('Demo_binary/260602_sub1_hjlee.xdf')

streams, _ = pyxdf.load_xdf(str(XDF_PATH), select_streams=[{'type': 'Markers'}])
len(streams)

In [ ]:
marker_rows = []
marker_counts = {}

for stream_idx, stream in enumerate(streams):
    info = stream.get('info', {})
    name = info.get('name', [''])[0]
    stream_type = info.get('type', [''])[0]
    values = np.asarray(stream['time_series'], dtype=object)

    if values.ndim == 1:
        flat_values = [str(value) for value in values]
    else:
        flat_values = ['|'.join(map(str, np.ravel(row))) for row in values]

    counts = Counter(flat_values)
    marker_counts[stream_idx] = counts
    for marker, count in counts.most_common():
        marker_rows.append({
            'stream_idx': stream_idx,
            'stream_name': name,
            'stream_type': stream_type,
            'marker': marker,
            'count': count,
        })

marker_count_df = pd.DataFrame(marker_rows)
marker_count_df

In [ ]:
trial_markers = ['1', '2', '3', '4', '5']
trial_markers_101 = ['101', '102', '103', '104', '105']

trial_rows = []
for stream_idx, counts in marker_counts.items():
    row = {'stream_idx': stream_idx}
    for marker in trial_markers:
        row[f'marker_{marker}'] = counts.get(marker, 0)
    row['total_1_to_5'] = sum(counts.get(marker, 0) for marker in trial_markers)
    for marker in trial_markers_101:
        row[f'marker_{marker}'] = counts.get(marker, 0)
    row['total_101_to_105'] = sum(counts.get(marker, 0) for marker in trial_markers_101)
    trial_rows.append(row)

trial_count_df = pd.DataFrame(trial_rows)
trial_count_df

In [ ]:
processed_260602 = [path for path in files if path.name == '260602_sub1_hjlee.npz']

if processed_260602:
    with np.load(processed_260602[0], allow_pickle=False) as data:
        processed_trials = len(data['y'])
else:
    processed_trials = None

pd.DataFrame([{
    'xdf_trial_count_1_to_5': int(trial_count_df['total_1_to_5'].sum()),
    'xdf_trial_count_101_to_105': int(trial_count_df['total_101_to_105'].sum()),
    'processed_260602_y_len': processed_trials,
}])

## Direct compare: XDF markers vs senior NPZ

This section does not assume the current preprocessing script created the NPZ. It compares `260602_sub1_hjlee.xdf` marker order directly against `260602_sub1_hjlee.npz` labels.

In [ ]:
import pyxdf

if 'DATA_DIR' not in globals():
    DATA_DIR = Path('data/processed')
    if not DATA_DIR.exists():
        DATA_DIR = Path('Demo_binary/data/processed')

if 'XDF_PATH' not in globals():
    XDF_PATH = Path('260602_sub1_hjlee.xdf')
    if not XDF_PATH.exists():
        XDF_PATH = Path('Demo_binary/260602_sub1_hjlee.xdf')

if 'streams' not in globals():
    streams, _ = pyxdf.load_xdf(str(XDF_PATH), select_streams=[{'type': 'Markers'}])

TARGET_NPZ = DATA_DIR / '260602_sub1_hjlee.npz'

with np.load(TARGET_NPZ, allow_pickle=False) as data:
    X_npz = data['X']
    y_npz = data['y'].astype(int)

trial_axis = [axis for axis, size in enumerate(X_npz.shape) if size == len(y_npz)]

pd.DataFrame([{
    'npz_file': TARGET_NPZ.name,
    'X_shape': X_npz.shape,
    'y_shape': y_npz.shape,
    'trial_axis_matching_y': trial_axis,
    'npz_trial_count': len(y_npz),
}])

In [ ]:
onset_rows = []

for stream_idx, stream in enumerate(streams):
    values = np.asarray(stream['time_series'], dtype=object)
    timestamps = np.asarray(stream['time_stamps'], dtype=float)

    if values.ndim == 1:
        flat_values = [str(value) for value in values]
    else:
        flat_values = ['|'.join(map(str, np.ravel(row))) for row in values]

    for marker_idx, (marker, timestamp) in enumerate(zip(flat_values, timestamps)):
        if marker in ['1', '2', '3', '4', '5']:
            onset_rows.append({
                'stream_idx': stream_idx,
                'marker_idx': marker_idx,
                'onset_idx': len(onset_rows),
                'marker': marker,
                'label': int(marker),
                'timestamp': timestamp,
            })

xdf_onsets = pd.DataFrame(onset_rows)
xdf_onsets.head(), xdf_onsets.tail(), len(xdf_onsets)

In [ ]:
xdf_labels = xdf_onsets['label'].to_numpy(dtype=int)
n_npz = len(y_npz)

match_rows = []
for start in range(0, len(xdf_labels) - n_npz + 1):
    segment = xdf_labels[start:start + n_npz]
    match_rows.append({
        'start_onset_idx': start,
        'end_onset_idx_exclusive': start + n_npz,
        'exact_match': bool(np.array_equal(segment, y_npz)),
        'mismatch_count': int(np.sum(segment != y_npz)),
    })

match_df = pd.DataFrame(match_rows).sort_values(['mismatch_count', 'start_onset_idx'])
match_df.head(10)

In [ ]:
best_start = int(match_df.iloc[0]['start_onset_idx'])
best_end = int(match_df.iloc[0]['end_onset_idx_exclusive'])
best_segment = xdf_labels[best_start:best_end]

compare_rows = []
for label in sorted(set(xdf_labels) | set(y_npz)):
    compare_rows.append({
        'label': label,
        'xdf_all_1_to_5': int(np.sum(xdf_labels == label)),
        'npz_y': int(np.sum(y_npz == label)),
        'xdf_best_matching_window': int(np.sum(best_segment == label)),
        'xdf_before_best_window': int(np.sum(xdf_labels[:best_start] == label)),
        'xdf_after_best_window': int(np.sum(xdf_labels[best_end:] == label)),
    })

pd.DataFrame(compare_rows)

In [ ]:
mismatch_idx = np.flatnonzero(best_segment != y_npz)

pd.DataFrame([{
    'best_start_onset_idx': best_start,
    'best_end_onset_idx_exclusive': best_end,
    'best_window_exact_match': bool(len(mismatch_idx) == 0),
    'best_window_mismatch_count': int(len(mismatch_idx)),
    'xdf_onsets_before_npz_match': best_start,
    'xdf_onsets_after_npz_match': int(len(xdf_labels) - best_end),
}])

In [ ]:
if len(mismatch_idx):
    pd.DataFrame({
        'relative_trial_idx': mismatch_idx[:30],
        'xdf_label': best_segment[mismatch_idx[:30]],
        'npz_y': y_npz[mismatch_idx[:30]],
    })
else:
    print('NPZ y exactly matches the XDF onset-label sequence in the best window.')

In [ ]:
if best_end < len(xdf_onsets):
    omitted_after = xdf_onsets.iloc[best_end:].copy()
    display(omitted_after['label'].value_counts().sort_index().rename('omitted_after_count').to_frame())
    display(omitted_after.head(10))
else:
    print('No XDF onset markers after the NPZ-matched window.')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(xdf_labels, '.', ms=3, label='XDF onset labels')
ax.plot(range(best_start, best_end), y_npz, '.', ms=2, label='NPZ y aligned to best window')
ax.axvline(best_start, color='black', lw=1, ls='--')
ax.axvline(best_end, color='black', lw=1, ls='--')
ax.set_xlabel('XDF onset index')
ax.set_ylabel('label')
ax.set_title('XDF marker sequence vs NPZ y sequence')
ax.legend()
plt.tight_layout()

### Optional direct signal check

Set `LOAD_EEG_SIGNAL = True` only when you want to load the full EEG stream from the 1.4 GB XDF. This compares selected NPZ trials against raw XDF EEG windows with the same onset timestamps.

In [ ]:
LOAD_EEG_SIGNAL = False

if LOAD_EEG_SIGNAL:
    eeg_streams, _ = pyxdf.load_xdf(str(XDF_PATH), select_streams=[{'type': 'EEG'}])
    eeg_stream = eeg_streams[0]
    eeg_values = np.asarray(eeg_stream['time_series'], dtype=float)
    eeg_timestamps = np.asarray(eeg_stream['time_stamps'], dtype=float)
    eeg_sfreq = float(eeg_stream['info']['nominal_srate'][0])

    pd.DataFrame([{
        'xdf_eeg_time_series_shape': eeg_values.shape,
        'xdf_eeg_timestamp_count': len(eeg_timestamps),
        'xdf_eeg_nominal_sfreq': eeg_sfreq,
        'npz_X_shape': X_npz.shape,
    }])
else:
    print('Set LOAD_EEG_SIGNAL = True to load the full EEG stream and compare signal windows.')

In [ ]:
if LOAD_EEG_SIGNAL:
    check_trials = [0, len(y_npz) // 2, len(y_npz) - 1]
    signal_rows = []

    for trial_idx in check_trials:
        onset_time = float(xdf_onsets.iloc[best_start + trial_idx]['timestamp'])
        xdf_start = int(np.searchsorted(eeg_timestamps, onset_time))
        xdf_stop = xdf_start + X_npz.shape[0]
        xdf_window = eeg_values[xdf_start:xdf_stop, :X_npz.shape[1]]
        npz_window = X_npz[:, :, trial_idx]

        if xdf_window.shape == npz_window.shape:
            corr = np.corrcoef(xdf_window.ravel(), npz_window.ravel())[0, 1]
            mae = np.mean(np.abs(xdf_window - npz_window))
        else:
            corr = np.nan
            mae = np.nan

        signal_rows.append({
            'trial_idx': trial_idx,
            'npz_label': int(y_npz[trial_idx]),
            'xdf_onset_idx': int(best_start + trial_idx),
            'xdf_start_sample': xdf_start,
            'xdf_window_shape': xdf_window.shape,
            'npz_window_shape': npz_window.shape,
            'raw_window_vs_npz_corr': corr,
            'raw_window_vs_npz_mae': mae,
        })

    pd.DataFrame(signal_rows)
else:
    print('Skipped. Set LOAD_EEG_SIGNAL = True in the previous cell.')